# Investigating model MCQ reasoning failures

In [ ]:
### Checks models MC reasoning failures

# %% Print MC Binding Failure Examples

import json
from pathlib import Path

# Configuration
MODEL_NAME = 'claude-opus-4-5-20251101'  # Change this to any model
TYPE = 'athletes'  # or 'custom'
BASE_DIR = f"./binding_failures_{TYPE}_temp0.3"
N_EXAMPLES = 5000  # Number of examples to show

model_path = Path(BASE_DIR) / MODEL_NAME.replace(':', '_')

print(f"\n{'='*80}")
print(f"Multi-Choice Binding Failures for: {MODEL_NAME}")
print(f"Dataset: {TYPE}")
print('='*80)

failure_count = 0

# Iterate through all fail_* directories
for fail_dir in sorted(model_path.glob("fail_*")):
    entity_name = fail_dir.name
    
    # Look for MC binding evaluation directory
    mc_bind_dir = fail_dir / "eval_binding_mc_reason"
    # mc_bind_dir = fail_dir / "eval_binding_mc_raw"

    
    if not mc_bind_dir.exists():
        continue
    
    # Check all evaluation files
    for eval_file in mc_bind_dir.glob("eval_*.json"):
        if eval_file.name.startswith("_debug"):
            continue
            
        with open(eval_file, 'r') as f:
            data = json.load(f)

        original_file = str(eval_file).replace('eval_', "")

        with open(original_file, 'r') as f:
            orig_data = json.load(f)
        
        # Check if it's a failure
        is_correct = data.get('correct', None)
        
        if not is_correct:
            failure_count += 1
            
            if failure_count <= N_EXAMPLES:
                print(f"\n{'─'*80}")
                print(f"Example {failure_count}: {entity_name}")
                print(f"File: {eval_file.name}")
                print('─'*80)
                
                # Print the query
                if 'query' in data:
                    print(f"\n📝 QUERY:\n{data['query']}")
                
                # Print the response
                if 'response' in data:
                    print(f"\n🤖 MODEL RESPONSE:\n{orig_data['response']}")
                
                # Print expected vs actual
                if 'expected' in data:
                    print(f"\n✓ EXPECTED: {data.get('expected', 'N/A')}")
                    print(f"✗ ACTUAL: {data.get('response', 'N/A')}")
                    if 'reasoning_trace' in orig_data:
                        print(f"\n💭 EVALUATION REASONING:\n{orig_data['reasoning_trace']}")
            
            if failure_count == N_EXAMPLES:
                break
    
    if failure_count == N_EXAMPLES:
        break

print(f"\n{'='*80}")
print(f"Total MC binding failures found: {failure_count}+")
print(f"Showing first {N_EXAMPLES} examples")
print('='*80)

# Investigating Reasoning failures

In [ ]:
from pathlib import Path
import json

BASE_DIR = "./binding_failures_athletes_temp0.3"
BIND_THRESH = 0.6

RAW = "binding_raw" # Instant in main paper
REASON = "binding_reason"

def load_eval_files(eval_dir):
    return sorted(f for f in eval_dir.glob("eval_*.json") if not f.name.startswith("_debug"))

def load_data_files(d_dir):
    return sorted(f for f in d_dir.glob("*.json") if not f.name.startswith("_debug"))

def is_correct(d):
    return d.get("correct", False) or d.get("evaluation", {}).get("correct", False)

def accuracy_and_example(eval_dir, d_dir, expect_correct):
    files = load_eval_files(eval_dir)
    data_files = load_data_files(d_dir)

    if not files or not data_files:
        return None, None

    correct = 0
    first_example = None

    for f,dt in zip(files, data_files):
        with open(f) as j:
            d = json.load(j)
        if is_correct(d):
            correct += 1
            if expect_correct and first_example is None:
                with open(dt) as j:
                    dta = json.load(j)
                first_example = (f, d, dta)
        elif not expect_correct and first_example is None:
            with open(dt) as j:
                dta = json.load(j)
            first_example = (f, d, dta)

    return correct / len(files), first_example


base = Path(BASE_DIR)

for model in sorted(d for d in base.iterdir() if d.is_dir()):
    for fail_dir in sorted(model.glob("fail_*")):
        raw_dir = fail_dir / f"eval_{RAW}"
        raw_dt_dir = fail_dir / RAW

        reason_dir = fail_dir / f"eval_{REASON}"
        reason_dt_dir = fail_dir / REASON

        if not raw_dir.exists() or not reason_dir.exists():
            continue

        raw_acc, raw_ex = accuracy_and_example(raw_dir,raw_dt_dir, True)
        reason_acc, reason_ex = accuracy_and_example(reason_dir,reason_dt_dir, False)

        if raw_acc is None or reason_acc is None:
            continue

        # condition: raw works, reasoning fails
        if raw_acc >= BIND_THRESH and reason_acc < BIND_THRESH and reason_ex is not None:
            reason_file, reason_data, reason_dt_data = reason_ex
            eval_id = reason_file.stem.split("_")[-1]

            # load corresponding RAW example (same id)
            raw_file, raw_data, raw_dt_data = raw_ex

            print("=" * 120)
            print(f"MODEL   : {model.name}")
            print(f"ENTITY  : {fail_dir.name}")
            print(f"ACCURACY: raw={raw_acc:.2f}  reason={reason_acc:.2f}")
            print("-" * 120)

            print("🟢 RAW BINDING RESPONSE:")
            print("Prompt: \n", raw_dt_data.get("prompt", ""))
            print("Answer: \n",raw_data.get("response", ""))
            print()

            print("🔴 REASONING BINDING RESPONSE:")
            print("Prompt: \n", reason_dt_data.get("prompt", ""))
            print("Answer: \n", reason_data.get("response", ""))
            print()

            print("EXPECTED:")
            print(reason_data.get("expected", ""))
            print("=" * 120, "\n")


# Investigating instant binding failures

In [ ]:
from pathlib import Path
import json

BASE_DIR = "./binding_failures_athletes_temp0.3"
BIND_THRESH = 0.6

RAW = "binding_raw"
REASON = "binding_reason"

def load_eval_files(eval_dir):
    return sorted(f for f in eval_dir.glob("eval_*.json") if not f.name.startswith("_debug"))

def load_data_files(d_dir):
    return sorted(f for f in d_dir.glob("*.json") if not f.name.startswith("_debug"))

def is_correct(d):
    return d.get("correct", False) or d.get("evaluation", {}).get("correct", False)

def accuracy_and_example(eval_dir, d_dir, expect_correct):
    files = load_eval_files(eval_dir)
    data_files = load_data_files(d_dir)

    if not files or not data_files:
        return None, None

    correct = 0
    first_example = None

    for f,dt in zip(files, data_files):
        with open(f) as j:
            d = json.load(j)
        if is_correct(d):
            correct += 1
            if expect_correct and first_example is None:
                with open(dt) as j:
                    dta = json.load(j)
                first_example = (f, d, dta)
        elif not expect_correct and first_example is None:
            with open(dt) as j:
                dta = json.load(j)
            first_example = (f, d, dta)

    return correct / len(files), first_example


base = Path(BASE_DIR)

for model in sorted(d for d in base.iterdir() if d.is_dir()):
    for fail_dir in sorted(model.glob("fail_*")):
        raw_dir = fail_dir / f"eval_{RAW}"
        raw_dt_dir = fail_dir / RAW

        reason_dir = fail_dir / f"eval_{REASON}"
        reason_dt_dir = fail_dir / REASON

        if not raw_dir.exists() or not reason_dir.exists():
            continue

        raw_acc, raw_ex = accuracy_and_example(raw_dir,raw_dt_dir, False)
        reason_acc, reason_ex = accuracy_and_example(reason_dir,reason_dt_dir, True)

        if raw_acc is None or reason_acc is None:
            continue

        # condition: raw works, reasoning fails
        if raw_acc < BIND_THRESH and reason_acc >= BIND_THRESH and reason_ex is not None:
            reason_file, reason_data, reason_dt_data = reason_ex
            eval_id = reason_file.stem.split("_")[-1]

            # load corresponding RAW example (same id)
            raw_file, raw_data, raw_dt_data = raw_ex

            print("=" * 120)
            print(f"MODEL   : {model.name}")
            print(f"ENTITY  : {fail_dir.name}")
            print(f"ACCURACY: raw={raw_acc:.2f}  reason={reason_acc:.2f}")
            print("-" * 120)

            print("🔴 RAW BINDING RESPONSE:")
            print("Prompt: \n", raw_dt_data.get("prompt", ""))
            print("Answer: \n",raw_data.get("response", ""))
            print()

            print("🟢 REASONING BINDING RESPONSE:")
            print("Prompt: \n", reason_dt_data.get("prompt", ""))
            print("Answer: \n", reason_data.get("response", ""))
            print()

            print("EXPECTED:")
            print(reason_data.get("expected", ""))
            print("=" * 120, "\n")
